# Linear regression

Predict the final grade as a continuous value using a leakage-aware mixed-type feature pipeline.

**Dataset:** `data/student_data.csv` (395 student records, 33 columns). **Target:** final grade `G3` (0–20). For classification labs, pass means `G3 ≥ 10`. Earlier grades `G1` and `G2` are included as predictors; interpret results as prediction after those earlier grades are known. The split is fixed with random seed 42 for reproducibility.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [Path("data/student_data.csv"), Path("../data/student_data.csv")]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place student_data.csv in the repository's data/ folder.")
df = pd.read_csv(DATA_PATH)
TARGET = "G3"  # final-period grade (0–20)
PASS_THRESHOLD = 10
numeric_features = df.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y_reg = df[TARGET]
y_class = (df[TARGET] >= PASS_THRESHOLD).astype(int)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def make_preprocessor(scale_numeric=True):
    numeric_steps = [("scale", StandardScaler())] if scale_numeric else []
    numeric_pipe = Pipeline(numeric_steps) if numeric_steps else "passthrough"
    return ColumnTransformer(
        [("numeric", numeric_pipe, numeric_features),
         ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="drop",
    )


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# G3 is excluded from X; G1 and G2 are retained as earlier-period grades.
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.25, random_state=42)
model = Pipeline([("preprocess", make_preprocessor()), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f"MAE:  {mean_absolute_error(y_test, pred):.2f} grade points")
print(f"RMSE: {mean_squared_error(y_test, pred) ** 0.5:.2f} grade points")
print(f"R²:   {r2_score(y_test, pred):.3f}")
plt.scatter(y_test, pred, alpha=.7)
plt.plot([0, 20], [0, 20], "--", color="black")
plt.xlabel("Actual G3"); plt.ylabel("Predicted G3"); plt.title("Linear regression: actual vs predicted")
plt.show()
